# Camada bronze - ingestão dos dados

Objetivo:
- Ler dados brutos (RAW)
- Padronizar estrutura
- Lidar com sensores ausentes
- Criar dataset unificado (Bronze)

Entrada: data/raw/Raw
Saída: data/bronze/bronze.parquet

# Importações

In [17]:
import os
import numpy as np
import pandas as pd
from mne.io import read_raw_brainvision

# Caminhos

In [18]:
RAW_PATH = "../data/raw/Raw"
BRONZE_PATH = "../data/bronze/bronze.parquet"

# Explorando sujeitos

In [19]:
subject = "001"
subject_path = os.path.join(RAW_PATH, subject)

os.listdir(subject_path)

['1211-003.eeg',
 '1211-003.vmrk',
 'LShank.csv',
 '1211-003.vhdr',
 'Arm.csv',
 'Waist.csv']

Como podemos ver o sujeito 1 apresenta arquivos nos formatos .eeg, .vmrk, .vhdr e csv's auxiliares, mas nem todos sujeitos possuem exatamente esses tipos de csv's e nem estão organizados da mesma forma:

In [20]:
subject = "004"
subject_path = os.path.join(RAW_PATH, subject)

os.listdir(subject_path)

['1219-007.eeg', '1219-007.vmrk', 'RShank.csv', 'Arm.csv', '1219-007.vhdr']

Isso faz com que tenhamos que padronizar esses dados brutos para a camada bronze que será a concatenação bruta deles.

# Lendo arquivos

In [21]:
vhdr_file = [f for f in os.listdir(subject_path) if f.endswith(".vhdr")][0]
vhdr_path = os.path.join(subject_path, vhdr_file)

raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)
eeg_data = raw.get_data().T

eeg_data.shape

/tmp/ipykernel_138300/4032978254.py:4: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)
/tmp/ipykernel_138300/4032978254.py:4: RuntimeWarning: Channels contain different lowpass filters. Highest (weakest) filter setting (500.00 Hz, Nyquist limit) will be stored.
  raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)


(911320, 32)

os.listdir(subject_path) lista os arquivos dentro da pasta do sujeito pra poder em seguida filtrar aqueles que tem o formato .vhdr, pegando assim o primeiro (e único) [0]. Por fim unimos o nome do arquivo com o caminho da pasta com o .join.

A função read_raw_brainvision da biblioteca do MNE vai carregar os dados EEG através do endereço do .vhdr, já que esse arquivo é o manual de instruções para o computador ler nosso arquivo .eeg sem mostrar diversas mensagens no terminal (verbose=False).

Depois usando a função get_data conseguimos os dados .eeg em uma matriz numérica transposta para o formato (tempo, canais), assim os dados brutos lidos corretamente com separação de canais. 

o .shape nos dará a dimensão da matriz final, nesse caso ela teve 911320 pontos e 32 canais, como nosso último sujeito explorado foi o 004, temos que ele teve então 911320 (sendo 1000 Hz a taxa de amostragem isso daria uns 15 minutos de gravações) amostras de tempo e os padrões 32 canais de EEG.



In [22]:
df = pd.DataFrame(eeg_data)
df.head(10)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,0.000000e+00,0.000002,-0.000005,0.000003,-0.000005,0.000002,-0.000013,0.0,-0.000016,-0.000008,...,-0.000003,0.0,-0.000004,0.000006,-0.000009,0.000004,0.000005,0.000044,0.000005,0.000012
1,5.000000e-07,0.000003,-0.000005,0.000004,-0.000005,0.000002,-0.000013,0.0,-0.000015,-0.000007,...,-0.000003,0.0,-0.000005,0.000007,-0.000008,0.000005,0.000002,0.000054,0.000005,0.000013
2,1.500000e-06,0.000004,-0.000005,0.000005,-0.000005,0.000002,-0.000012,0.0,-0.000014,-0.000006,...,-0.000003,0.0,-0.000005,0.000008,-0.000007,0.000005,0.000002,0.000062,0.000005,0.000013
3,2.500000e-06,0.000005,-0.000005,0.000005,-0.000005,0.000002,-0.000012,0.0,-0.000013,-0.000005,...,-0.000003,0.0,-0.000005,0.000008,-0.000007,0.000006,0.000000,0.000063,0.000005,0.000013
4,3.500000e-06,0.000005,-0.000005,0.000005,-0.000005,0.000003,-0.000011,0.0,-0.000012,-0.000003,...,-0.000003,0.0,-0.000005,0.000009,-0.000006,0.000006,0.000000,0.000065,0.000005,0.000013
5,4.000000e-06,0.000006,-0.000005,0.000005,-0.000005,0.000003,-0.000010,0.0,-0.000010,-0.000002,...,-0.000003,0.0,-0.000005,0.000009,-0.000006,0.000006,0.000016,0.000080,0.000005,0.000013
6,4.500000e-06,0.000005,-0.000005,0.000005,-0.000005,0.000003,-0.000009,0.0,-0.000009,0.000000,...,-0.000003,0.0,-0.000005,0.000010,-0.000005,0.000007,0.000032,0.000090,0.000006,0.000013
7,4.500000e-06,0.000005,-0.000005,0.000004,-0.000004,0.000004,-0.000008,0.0,-0.000007,0.000002,...,-0.000003,0.0,-0.000005,0.000010,-0.000005,0.000007,0.000035,0.000089,0.000007,0.000013
8,5.000000e-06,0.000005,-0.000005,0.000004,-0.000004,0.000005,-0.000008,0.0,-0.000005,0.000003,...,-0.000003,0.0,-0.000005,0.000011,-0.000004,0.000007,0.000035,0.000084,0.000008,0.000013
9,5.000000e-06,0.000003,-0.000005,0.000004,-0.000003,0.000005,-0.000007,0.0,-0.000004,0.000005,...,-0.000002,0.0,-0.000003,0.000011,-0.000003,0.000007,0.000032,0.000083,0.000009,0.000013


Acima temos as 10 primeiras linhas do dataframe dos sinais EEG do sujeito 4.

Vamos agora tratar os csv's do sujeito 004:

In [23]:
def load_sensor(subject_path, sensor):
    file_path = os.path.join(subject_path, f"{sensor}.csv")
    
    if os.path.exists(file_path):
        data = pd.read_csv(file_path, header=None)
        data = data.iloc[:, 1:7]
        
        data.columns = [
            f"{sensor}_ACC_X",
            f"{sensor}_ACC_Y",
            f"{sensor}_ACC_Z",
            f"{sensor}_GYRO_X",
            f"{sensor}_GYRO_Y",
            f"{sensor}_GYRO_Z",
        ]
        
        return data
    else:
        return None

No código acima temos o endereço pro sensor sendo criado, em seguida temos a leitura do arquivo csv pegando apenas as colunas da posição 1 até a 6 (ignora a 0 que é o tempo). Depois colocamos nomes em cada uma das 6 colunas daquele sensor, 3 pro acelerômetro e 3 pro giroscópio, cada uma sendo um eixo. 

Verificando pro sensor arm do sujeito 4:

In [24]:
sensor_data = load_sensor(subject_path, "Arm")

if sensor_data is not None:
    print("Sucesso! O formato dos dados é:", sensor_data.shape)
    print(sensor_data.head()) # Mostra as primeiras 5 linhas com os nomes das colunas
else:
    print("Vish! O arquivo não foi encontrado no caminho especificado.")

Sucesso! O formato dos dados é: (224654, 6)
   Arm_ACC_X  Arm_ACC_Y  Arm_ACC_Z  Arm_GYRO_X  Arm_GYRO_Y  Arm_GYRO_Z
0        151       1164       7647          52          50           6
1        145       1153       7638          51          46           7
2        145       1153       7638          51          46           7
3        145       1153       7638          51          46           7
4        145       1153       7638          51          46           7


Se fossemos tentar achar pro sensor LShank, não iríamos encontrar nada pois o sujeito 4 não usou esse sensor: 

In [25]:
sensor_data = load_sensor(subject_path, "LShank")

if sensor_data is not None:
    print("Sucesso! O formato dos dados é:", sensor_data.shape)
    print(sensor_data.head()) # Mostra as primeiras 5 linhas com os nomes das colunas
else:
    print("Vish! O arquivo não foi encontrado no caminho especificado.")

Vish! O arquivo não foi encontrado no caminho especificado.


Agora, como podemos perceber acima, o número de medições é diferente pro mesmo sujeito: sensores-> (224654, 6), eeg-> (911320, 32). Logo temos que percorrer por todos os tipos de sensores que um sujeito pode ter, usar o load_sensor que fizemos e ir pegando o menor número de medições dentre os sensores pra cortar as linhas extras e se um determinado sensor não existir pra um sujeito ele vai ter suas colunas preenchidas com NaN, depois, concatenamos tudo isso em uma grande tabela. implementamos isso da seguinte forma:

In [26]:
sensors = ["LShank", "RShank", "Waist", "Arm"]

for sensor in sensors:
    sensor_df = load_sensor(subject_path, sensor)
    
    if sensor_df is not None:
        min_len = min(len(df), len(sensor_df))
        df = df.iloc[:min_len]
        sensor_df = sensor_df.iloc[:min_len]
        
        df = pd.concat([df, sensor_df], axis=1)
    else:
        for col in [
            f"{sensor}_ACC_X",
            f"{sensor}_ACC_Y",
            f"{sensor}_ACC_Z",
            f"{sensor}_GYRO_X",
            f"{sensor}_GYRO_Y",
            f"{sensor}_GYRO_Z",
        ]:
            df[col] = np.nan

Obtemos então a seguinte tabela de sinais para o sujeito 004:

In [27]:
df["subject"] = subject
print("Concatenação foi um sucesso! O formato dos dados é:", df.shape)
df.head()


Concatenação foi um sucesso! O formato dos dados é: (224654, 57)


,0,1,2,3,4,5,6,7,8,9,...,Waist_GYRO_X,Waist_GYRO_Y,Waist_GYRO_Z,Arm_ACC_X,Arm_ACC_Y,Arm_ACC_Z,Arm_GYRO_X,Arm_GYRO_Y,Arm_GYRO_Z,subject
0,0.000000e+00,0.000002,-0.000005,0.000003,-0.000005,0.000002,-0.000013,0.0,-0.000016,-0.000008,...,NaN,NaN,NaN,151,1164,7647,52,50,6,004
1,5.000000e-07,0.000003,-0.000005,0.000004,-0.000005,0.000002,-0.000013,0.0,-0.000015,-0.000007,...,NaN,NaN,NaN,145,1153,7638,51,46,7,004
2,1.500000e-06,0.000004,-0.000005,0.000005,-0.000005,0.000002,-0.000012,0.0,-0.000014,-0.000006,...,NaN,NaN,NaN,145,1153,7638,51,46,7,004
3,2.500000e-06,0.000005,-0.000005,0.000005,-0.000005,0.000002,-0.000012,0.0,-0.000013,-0.000005,...,NaN,NaN,NaN,145,1153,7638,51,46,7,004
4,3.500000e-06,0.000005,-0.000005,0.000005,-0.000005,0.000003,-0.000011,0.0,-0.000012,-0.000003,...,NaN,NaN,NaN,145,1153,7638,51,46,7,004


# Organizando os sujeitos

In [28]:
all_data = []

for subject in os.listdir(RAW_PATH):
    subject_path = os.path.join(RAW_PATH, subject)
    
    if not os.path.isdir(subject_path):
        continue
    
    # detectar subpastas, caso seja o 008 por exemplo
    subfolders = [f for f in os.listdir(subject_path) 
                  if os.path.isdir(os.path.join(subject_path, f))]
    
    # definir caminhos a processar
    if len(subfolders) == 0:
        paths_to_process = [subject_path]
    else:
        paths_to_process = [
            os.path.join(subject_path, sub) for sub in subfolders
        ]
    
    # loop nas sessões (ou sujeito único)
    for path in paths_to_process:
        
        vhdr_files = [f for f in os.listdir(path) if f.endswith(".vhdr")]
        
        if not vhdr_files:
            continue
        
        vhdr_path = os.path.join(path, vhdr_files[0])
        
        raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)
        eeg_data = raw.get_data().T
        
        df = pd.DataFrame(eeg_data)
        
        # sensores 
        for sensor in sensors:
            sensor_df = load_sensor(path, sensor) 
            
            if sensor_df is not None:
                min_len = min(len(df), len(sensor_df))
                df = df.iloc[:min_len]
                sensor_df = sensor_df.iloc[:min_len]
                
                df = pd.concat([df, sensor_df], axis=1)
            else:
                for col in [
                    f"{sensor}_ACC_X",
                    f"{sensor}_ACC_Y",
                    f"{sensor}_ACC_Z",
                    f"{sensor}_GYRO_X",
                    f"{sensor}_GYRO_Y",
                    f"{sensor}_GYRO_Z",
                ]:
                    df[col] = np.nan
        
        # metadados
        df["subject"] = subject
        
        # se tiver subpasta → é sessão
        if len(subfolders) > 0:
            df["session"] = os.path.basename(path)
        else:
            df["session"] = "1"
        
        all_data.append(df)

/tmp/ipykernel_138300/665623010.py:31: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)
/tmp/ipykernel_138300/665623010.py:31: RuntimeWarning: Channels contain different lowpass filters. Highest (weakest) filter setting (500.00 Hz, Nyquist limit) will be stored.
  raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)
/tmp/ipykernel_138300/665623010.py:31: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)
/tmp/ipykernel_138300/665623010.py:31: RuntimeWarning: Channels contain different lowpass filters. Highest (weakest) filter setting (500.00 Hz, Nyquist limit) will be stored.
  raw = read_raw_brainvision(vhdr_path, preload=True, verbose=False)
/tmp/ipykernel_138300/665623010.py:31: RuntimeWarning: Online software filter 

Acima, começamos percorrendo as pastas de cada pessoa no RAW_PATH, i.e., sujeito 001, 002, 003... (processamento em lote). Para cada sujeito, também verificamos se existem subpastas internas (sessões/experimentos), como no caso do sujeito 008, e percorremos cada uma delas individualmente.

Repetimos o processo que já fizemos acima, de pegar o .vhdr, que é o arquivo de cabeçalho. Para cada sujeito (e, quando aplicável, para cada sessão), o sinal EEG é carregado e transposto para o formato de (amostras, canais). Em seguida, um laço interno vai pegar os sensores de interesse, alinhando temporalmente por truncamento para cortar as sobras.

Cada linha do DataFrame recebe uma etiqueta com o ID do sujeito e, quando existe, a identificação da sessão, e todas as tabelas individuais são armazenadas em uma lista chamada all_data, para posterior concatenação global.

Juntando todos os sujeitos:

# Criando camada bronze

In [29]:
bronze_df = pd.concat(all_data, ignore_index=True)
bronze_df.shape

(3872933, 58)

Obtemos mais de 3 milhões de linhas de dados de todos os sujeitos, com cerca de 58 colunas 

In [30]:
from pandas.io.parquet import to_parquet

os.makedirs("data/bronze", exist_ok=True)
bronze_df.to_parquet(BRONZE_PATH)

/home/natan/Área de trabalho/TI093-biosignals-acquisition/venv/lib/python3.12/site-packages/pandas/io/parquet.py:191: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


In [31]:
bronze_df.info()
bronze_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 3872933 entries, 0 to 3872932
Data columns (total 58 columns):
 #   Column         Dtype  
---  ------         -----  
 0   0              float64
 1   1              float64
 2   2              float64
 3   3              float64
 4   4              float64
 5   5              float64
 6   6              float64
 7   7              float64
 8   8              float64
 9   9              float64
 10  10             float64
 11  11             float64
 12  12             float64
 13  13             float64
 14  14             float64
 15  15             float64
 16  16             float64
 17  17             float64
 18  18             float64
 19  19             float64
 20  20             float64
 21  21             float64
 22  22             float64
 23  23             float64
 24  24             float64
 25  25             float64
 26  26             float64
 27  27             float64
 28  28             float64
 29  29             float64
 3

,0,1,2,3,4,5,6,7,8,9,...,Waist_GYRO_Y,Waist_GYRO_Z,Arm_ACC_X,Arm_ACC_Y,Arm_ACC_Z,Arm_GYRO_X,Arm_GYRO_Y,Arm_GYRO_Z,subject,session
0,-0.000020,-0.000002,-0.000005,0.000005,0.000020,0.000005,0.000006,-0.000007,-0.00001,-0.000015,...,57.0,24.0,NaN,NaN,NaN,NaN,NaN,NaN,002,1
1,-0.000020,-0.000002,-0.000005,0.000005,0.000019,0.000005,0.000005,-0.000007,-0.00001,-0.000015,...,57.0,23.0,NaN,NaN,NaN,NaN,NaN,NaN,002,1
2,-0.000019,-0.000002,-0.000005,0.000005,0.000019,0.000005,0.000005,-0.000007,-0.00001,-0.000015,...,58.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,002,1
3,-0.000018,-0.000002,-0.000005,0.000005,0.000019,0.000005,0.000005,-0.000007,-0.00001,-0.000015,...,58.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,002,1
4,-0.000018,-0.000002,-0.000005,0.000005,0.000019,0.000005,0.000005,-0.000007,-0.00001,-0.000015,...,58.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,002,1


In [32]:
for i, col in enumerate(bronze_df.columns):
    print(i, col)

0 0
1 1
2 2
3 3
4 4
5 5
6 6
7 7
8 8
9 9
10 10
11 11
12 12
13 13
14 14
15 15
16 16
17 17
18 18
19 19
20 20
21 21
22 22
23 23
24 24
25 25
26 26
27 27
28 28
29 29
30 30
31 31
32 LShank_ACC_X
33 LShank_ACC_Y
34 LShank_ACC_Z
35 LShank_GYRO_X
36 LShank_GYRO_Y
37 LShank_GYRO_Z
38 RShank_ACC_X
39 RShank_ACC_Y
40 RShank_ACC_Z
41 RShank_GYRO_X
42 RShank_GYRO_Y
43 RShank_GYRO_Z
44 Waist_ACC_X
45 Waist_ACC_Y
46 Waist_ACC_Z
47 Waist_GYRO_X
48 Waist_GYRO_Y
49 Waist_GYRO_Z
50 Arm_ACC_X
51 Arm_ACC_Y
52 Arm_ACC_Z
53 Arm_GYRO_X
54 Arm_GYRO_Y
55 Arm_GYRO_Z
56 subject
57 session
